# Auditoria RS 2024 (5ª Fase): Bacias + Vento Barra RG + ANA + Infra + SIH/SUS

Notebook pronto para execução no **Google Colab** usando a arquitetura modular em `src/`.

- **Vento (INMET / BigQuery):** decomposição vetorial horária e represamento na Barra de Rio Grande (`A802`).
- **Cotas Fluviométricas (ANA HidroWebService):** séries diárias e telemetria em tempo real (com fallback gracioso caso sem token).
- **Falhas de Infraestrutura:** eventos binários em `data/eventos_infraestrutura_rs.csv` (diques e casas de bombas) com ramo monotônico estrutural.
- **SIH/SUS & CNES-SUS:** deduplicação por `id_aih`, agrupamento por `data_internacao`, sem Dengue (`A90–A92`) e alinhamento estrito SUS vs. SUS.

In [ ]:
# 1. Verifica GPU, clona o repositório e instala apenas dependências faltantes no Python do Colab
import os
import sys
from pathlib import Path

!nvidia-smi || true

MOUNT_DRIVE = True  # @param {type:"boolean"}
REPO_URL = "https://github.com/GuilhermeSPedroso07/Previsao-de-leitos.git"
REPO_DIR = Path("/content/Previsao-de-leitos-master")

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA = Path("/content/drive/MyDrive/PrevisaoLeitos/data")
    DRIVE_DATA.mkdir(parents=True, exist_ok=True)
    os.environ["DATA_DIR"] = str(DRIVE_DATA)
    print(f"[DRIVE] DATA_DIR configurado para: {DRIVE_DATA}")

if not REPO_DIR.exists():
    !git clone "{REPO_URL}" "{REPO_DIR}"
os.chdir(REPO_DIR)
print("[REPO] Diretório atual:", os.getcwd())

# Reutiliza PyTorch/CUDA/pandas já pré-instalados no Colab e instala o que faltar + src/
!"{sys.executable}" -m pip install -q python-dotenv db-dtypes requests google-cloud-bigquery
!"{sys.executable}" -m pip install -q -e . --no-deps

In [ ]:
# 2. Autenticação GCP + Leitura de Secrets do Colab (GCP_PROJECT_ID, ANA_IDENTIFICADOR, ANA_SENHA)
import os
from google.colab import auth, userdata

auth.authenticate_user()

for key in ("GCP_PROJECT_ID", "ANA_IDENTIFICADOR", "ANA_SENHA", "ANA_API_BASE_URL"):
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = val.strip()
    except Exception:
        pass

assert os.environ.get("GCP_PROJECT_ID"), "Defina GCP_PROJECT_ID nos Secrets do Colab (ícone de chave à esquerda)."
print("GCP_PROJECT_ID:", os.environ["GCP_PROJECT_ID"])
print("Credenciais ANA configuradas:", bool(os.environ.get("ANA_IDENTIFICADOR") and os.environ.get("ANA_SENHA")))

In [ ]:
# 3. Consulta rápida de telemetria ANA em tempo real + inspeção do catálogo de falhas de infraestrutura
import os
import sys
import pandas as pd

os.chdir("/content/Previsao-de-leitos-master")
!PYTHONPATH=. "{sys.executable}" -m src.run_audit_2024 --ana-realtime-only

df_infra = pd.read_csv("data/eventos_infraestrutura_rs.csv")
print(f"\nEventos de infraestrutura carregados ({len(df_infra)} registros):")
display(df_infra)

In [ ]:
# 4. Execução completa da Auditoria RS 2024 com salvamento de pesos e gráficos
import glob
import os
import sys
from pathlib import Path
from IPython.display import Image, display

os.chdir("/content/Previsao-de-leitos-master")
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
PLOTS_DIR = DATA_DIR / "plots"
MODEL_PATH = DATA_DIR / "hydro_film_basin_rs.pt"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Para reutilizar pesos já treinados sem retreinar, troque --save-model por --load-model
!PYTHONPATH=. "{sys.executable}" -m src.run_audit_2024 --epochs 10 --save-plots-dir "{PLOTS_DIR}" --save-model "{MODEL_PATH}"

for img_path in sorted(glob.glob(str(PLOTS_DIR / "*.png"))):
    print(img_path)
    display(Image(filename=img_path))